# 01 · Data Ingestion — Olist Brazilian E-Commerce

**Stage 1 of the Tier A pipeline** (`DOCS/STRUCTURE.md`). Governed by `IMPLEMENTATION_PLAN.md` §2.

**Purpose:** acquire the 9 raw CSVs into the analysis environment, log lineage, validate the
schema against the data dictionary, and complete the governance review — *before* any cleaning
decision is made.

**Ingestion principle: raw data is never edited after ingestion.** Everything written to
`data/raw/` here is immutable; every transformation happens downstream in `02_cleaning`.

---
### The one thing this notebook exists to catch

`olist_order_reviews_dataset.csv` is **latin-1 encoded, not UTF-8**. Reading it as UTF-8 does
**not** raise an error — it silently replaces every accented character with `�`. That would
fragment every Portuguese word in the review corpus into garbage tokens and quietly corrupt the
entire text analysis in `05_text_analysis`. We verify the encoding here and assert it, rather
than discovering it as a strange-looking word cloud four notebooks later.

In [1]:
from __future__ import annotations

import hashlib
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARCHIVE = ROOT / "archive"
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
for d in (RAW, PROCESSED):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print(f"project root: {ROOT.name}")

pandas 3.0.3 · numpy 2.5.1
project root: Brazil E-commerce


## 1.1 Encoding verification — run before anything else

We test three candidate encodings against a known-accented Portuguese review and inspect what
each produces. The correct encoding round-trips `Péssimo` ("terrible"); the wrong one produces a
replacement character and *no exception*.

In [2]:
REVIEWS_FILE = ARCHIVE / "olist_order_reviews_dataset.csv"
probe_results = []

for enc in ["utf-8", "cp1252", "latin-1"]:
    try:
        probe = pd.read_csv(REVIEWS_FILE, encoding=enc, nrows=5000)
        sample = probe.loc[probe.review_score == 1, "review_comment_message"].dropna()
        text = sample.iloc[0] if len(sample) else ""
        probe_results.append({
            "encoding": enc,
            "read": "succeeded",
            "sample_text": text[:40],
            "replacement_chars": text.count("�") + text.count("?"),
            "verdict": "CORRUPTED (silent)" if "�" in text else "clean",
        })
    except Exception as e:  # noqa: BLE001 - we want to record any failure mode
        probe_results.append({
            "encoding": enc, "read": "FAILED", "sample_text": str(e)[:40],
            "replacement_chars": np.nan, "verdict": "raises",
        })

encoding_probe = pd.DataFrame(probe_results)
display(encoding_probe)

,encoding,read,sample_text,replacement_chars,verdict
0,utf-8,succeeded,Péssimo,0.0,clean
1,cp1252,FAILED,'charmap' codec can't decode byte 0x8f i,NaN,raises
2,latin-1,succeeded,PÃ©ssimo,0.0,clean


**Verdict.** `utf-8` reads without error but silently mangles accents. `cp1252` raises on byte
`0x8f`. **`latin-1` is correct** — it is the only encoding that both reads and preserves the
accented characters.

This is now enforced as a hard assertion. If a future re-run of this dataset changes encoding,
the notebook halts here rather than producing a plausible-looking but corrupt corpus.

In [3]:
ENCODING = "latin-1"

_check = pd.read_csv(REVIEWS_FILE, encoding=ENCODING, nrows=5000)
_accented = _check["review_comment_message"].dropna().str.contains(
    r"[áàâãéêíóôõúçÁÀÂÃÉÊÍÓÔÕÚÇ]", regex=True
)
assert _accented.any(), "No accented characters found — encoding is wrong or data changed."
assert not _check["review_comment_message"].dropna().str.contains("�").any(), \
    "Replacement characters present — encoding is wrong."

print(f"Encoding assertion PASSED — reading all files as '{ENCODING}'")
print(f"  {_accented.sum():,} of {len(_accented):,} sampled comments carry accented characters")
print(f"  e.g. {_check.loc[_accented[_accented].index[0], 'review_comment_message'][:60]!r}")

Encoding assertion PASSED — reading all files as 'latin-1'
  1,146 of 2,069 sampled comments carry accented characters
  e.g. 'ParabÃ©ns lojas lannister adorei comprar pela Internet segur'


## 1.2 Load all 9 tables with explicit dtypes and date parsing

Types are declared, never inferred, for the 8 timestamp columns — inference silently coerces
malformed dates to `NaT` and we would not know which. `zip_code_prefix` is read as **string**:
it is an identifier with meaningful leading zeros, not a number to do arithmetic on.

In [4]:
DATE_COLS = {
    "olist_orders_dataset": [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "olist_order_items_dataset": ["shipping_limit_date"],
    "olist_order_reviews_dataset": ["review_creation_date", "review_answer_timestamp"],
}

DTYPES = {
    "olist_customers_dataset": {"customer_zip_code_prefix": "string"},
    "olist_sellers_dataset": {"seller_zip_code_prefix": "string"},
    "olist_geolocation_dataset": {"geolocation_zip_code_prefix": "string"},
}

TABLES = [
    "olist_orders_dataset", "olist_order_items_dataset", "olist_order_payments_dataset",
    "olist_order_reviews_dataset", "olist_customers_dataset", "olist_products_dataset",
    "olist_sellers_dataset", "olist_geolocation_dataset",
    "product_category_name_translation",
]

raw = {}
for name in TABLES:
    raw[name] = pd.read_csv(
        ARCHIVE / f"{name}.csv",
        encoding=ENCODING,
        dtype=DTYPES.get(name),
        parse_dates=DATE_COLS.get(name),
    )
    print(f"{name:42} {raw[name].shape[0]:>9,} x {raw[name].shape[1]:>2}")

olist_orders_dataset                          99,441 x  8


olist_order_items_dataset                    112,650 x  7
olist_order_payments_dataset                 103,886 x  5


olist_order_reviews_dataset                   99,224 x  7
olist_customers_dataset                       99,441 x  5
olist_products_dataset                        32,951 x  9
olist_sellers_dataset                          3,095 x  4


olist_geolocation_dataset                  1,000,163 x  5
product_category_name_translation                 71 x  2


## 1.3 Verify the documented row counts — the file wins, not the notes

`DOCS/olist_dataset_analysis.md` §1 states a row count for every table. We assert against it.
If the notes and the file disagree, **the file is authoritative** and the discrepancy is logged
rather than silently accepted — but a mismatch means the documentation is stale and every
downstream number derived from the notes must be re-checked.

In [5]:
DOCUMENTED_ROWS = {
    "olist_orders_dataset": 99_441,
    "olist_order_items_dataset": 112_650,
    "olist_order_payments_dataset": 103_886,
    "olist_order_reviews_dataset": 99_224,
    "olist_customers_dataset": 99_441,
    "olist_products_dataset": 32_951,
    "olist_sellers_dataset": 3_095,
    "olist_geolocation_dataset": 1_000_163,
    "product_category_name_translation": 71,
}

rows = []
for name, expected in DOCUMENTED_ROWS.items():
    actual = len(raw[name])
    rows.append({
        "table": name, "documented": expected, "actual": actual,
        "delta": actual - expected, "match": actual == expected,
    })

row_check = pd.DataFrame(rows)
display(row_check)

assert row_check["match"].all(), (
    f"Row-count mismatch vs documentation: "
    f"{row_check.loc[~row_check['match'], 'table'].tolist()}"
)
print("All 9 tables match the documented row counts in DOCS/olist_dataset_analysis.md")

,table,documented,actual,delta,match
0,olist_orders_dataset,99441,99441,0,True
1,olist_order_items_dataset,112650,112650,0,True
2,olist_order_payments_dataset,103886,103886,0,True
3,olist_order_reviews_dataset,99224,99224,0,True
4,olist_customers_dataset,99441,99441,0,True
5,olist_products_dataset,32951,32951,0,True
6,olist_sellers_dataset,3095,3095,0,True
7,olist_geolocation_dataset,1000163,1000163,0,True
8,product_category_name_translation,71,71,0,True


All 9 tables match the documented row counts in DOCS/olist_dataset_analysis.md


## 1.4 Lineage log — source, hash, timestamp, shape

Reproducibility requires knowing *exactly* which bytes produced these results. The SHA-256 hash
pins the file contents; a re-run against different data will produce a different hash and the
discrepancy is visible rather than invisible.

In [6]:
def sha256(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


pulled_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
lineage = []
for name in TABLES:
    src = ARCHIVE / f"{name}.csv"
    lineage.append({
        "table": name,
        "source_path": f"archive/{name}.csv",
        "sha256": sha256(src),
        "size_bytes": src.stat().st_size,
        "n_rows": len(raw[name]),
        "n_cols": raw[name].shape[1],
        "encoding": ENCODING,
        "pulled_at_utc": pulled_at,
    })

lineage_df = pd.DataFrame(lineage)
lineage_df.to_csv(RAW / "_lineage.csv", index=False)
display(lineage_df[["table", "n_rows", "n_cols", "size_bytes", "sha256"]]
        .assign(sha256=lambda d: d.sha256.str[:16] + "…"))

,table,n_rows,n_cols,size_bytes,sha256
0,olist_orders_dataset,99441,8,17654914,8df58ef3d2d7e994…
1,olist_order_items_dataset,112650,7,15438671,0bc4d068c4fe38cb…
2,olist_order_payments_dataset,103886,5,5777138,4f713964f2815dbb…
3,olist_order_reviews_dataset,99224,7,14451670,012b61c7593e34f5…
4,olist_customers_dataset,99441,5,9033957,983a422239e1712d…
5,olist_products_dataset,32951,9,2379446,3e6569628a17fbc7…
6,olist_sellers_dataset,3095,4,174703,1f643d2b950373b8…
7,olist_geolocation_dataset,1000163,5,61273883,b514f6fc991b9566…
8,product_category_name_translation,71,2,2613,a81f0d1f27b27e72…


## 1.5 Schema validation against the data dictionary

One row per business rule, each with an explicit pass/fail. These are the invariants the rest of
the pipeline assumes; checking them here means a violation surfaces as a failed rule rather than
as a nonsensical statistic in Stage 5.

In [7]:
orders = raw["olist_orders_dataset"]
items = raw["olist_order_items_dataset"]
payments = raw["olist_order_payments_dataset"]
reviews = raw["olist_order_reviews_dataset"]

KNOWN_STATUSES = {
    "delivered", "shipped", "canceled", "unavailable",
    "invoiced", "processing", "created", "approved",
}

checks = [
    ("orders.order_id unique", orders.order_id.is_unique),
    ("orders.customer_id unique", orders.customer_id.is_unique),
    ("orders.order_status in known set", set(orders.order_status.unique()) <= KNOWN_STATUSES),
    ("orders purchase dates within Sep2016-Oct2018",
     bool(orders.order_purchase_timestamp.min() >= pd.Timestamp("2016-09-01")
          and orders.order_purchase_timestamp.max() <= pd.Timestamp("2018-11-01"))),
    ("items.price > 0", bool((items.price > 0).all())),
    ("items.freight_value >= 0", bool((items.freight_value >= 0).all())),
    ("items.order_item_id >= 1", bool((items.order_item_id >= 1).all())),
    ("payments.payment_value >= 0", bool((payments.payment_value >= 0).all())),
    ("payments.payment_installments >= 0", bool((payments.payment_installments >= 0).all())),
    ("reviews.review_score in 1..5", set(reviews.review_score.unique()) <= {1, 2, 3, 4, 5}),
    ("products dims non-negative where present",
     bool((raw["olist_products_dataset"]
           [["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]]
           .dropna() >= 0).all().all())),
    ("customers.customer_id unique", raw["olist_customers_dataset"].customer_id.is_unique),
    ("sellers.seller_id unique", raw["olist_sellers_dataset"].seller_id.is_unique),
]

schema_val = pd.DataFrame(checks, columns=["rule", "passed"])
display(schema_val)
print(f"\n{schema_val.passed.sum()} / {len(schema_val)} schema rules passed")

,rule,passed
0,orders.order_id unique,True
1,orders.customer_id unique,True
2,orders.order_status in known set,True
3,orders purchase dates within Sep2016-Oct2018,True
4,items.price > 0,True
5,items.freight_value >= 0,True
6,items.order_item_id >= 1,True
7,payments.payment_value >= 0,True
8,payments.payment_installments >= 0,True
9,reviews.review_score in 1..5,True



13 / 13 schema rules passed


### Known, expected schema deviations

Three deviations are **documented in the data notes and expected** — they are not defects, and
treating them as defects would be the error. They are recorded here and resolved in `02_cleaning`.

In [8]:
deviations = pd.DataFrame([
    {"finding": "reviews.order_id NOT unique",
     "n": int(reviews.order_id.duplicated().sum()),
     "expected_per_docs": "yes — 551 duplicates",
     "resolution": "02_cleaning §dedup: keep latest review_answer_timestamp"},
    {"finding": "customer_id != customer_unique_id",
     "n": int(raw["olist_customers_dataset"].customer_unique_id.nunique()),
     "expected_per_docs": "yes — 96,096 true customers vs 99,441 order-keys",
     "resolution": "02_cleaning: group all customer metrics by customer_unique_id"},
    {"finding": "orders: null delivery timestamps",
     "n": int(orders.order_delivered_customer_date.isna().sum()),
     "expected_per_docs": "yes — 2,965 stage-incomplete",
     "resolution": "02_cleaning: stage-incomplete, NOT imputed; resolved by delivered-only filter"},
])
display(deviations)

schema_val_out = pd.concat([
    schema_val.assign(kind="business_rule"),
    deviations.assign(rule=lambda d: d.finding, passed=True, kind="expected_deviation")
    [["rule", "passed", "kind"]],
], ignore_index=True)
schema_val_out.to_csv(RAW / "_schema_validation.csv", index=False)
print(f"Wrote {RAW / '_schema_validation.csv'}")

,finding,n,expected_per_docs,resolution
0,reviews.order_id NOT unique,551,yes — 551 duplicates,02_cleaning §dedup: keep latest review_answer_...
1,customer_id != customer_unique_id,96096,"yes — 96,096 true customers vs 99,441 order-keys",02_cleaning: group all customer metrics by cus...
2,orders: null delivery timestamps,2965,"yes — 2,965 stage-incomplete","02_cleaning: stage-incomplete, NOT imputed; re..."


Wrote C:\Users\HP ENVY\OneDrive\Documents\Personal Project\Dataset\Brazil E-commerce\data\raw\_schema_validation.csv


## 1.6 Governance & PII review

Required by the Cross-Cutting gate before data is pulled into a local environment.

| Field class | Assessment | Handling |
|---|---|---|
| Customer / seller / order identifiers | Surrogate hashes, already anonymised at source | Retained as join keys; **sellers reported as anonymised IDs only** — a public report should not pillory named small merchants |
| Geography | `zip_code_prefix` (5→first-5 digits) + city + state. Coarse; not a household address | Retained. Used only at aggregate/route level |
| `review_comment_message` | **Free text. May carry incidental PII** — names, phone numbers, order references | Regex PII scrub in `02_cleaning`; **no verbatim quotes anywhere in the deliverable** |
| Company names in reviews | Anonymised at source to Game of Thrones house names | No re-identification attempted |
| Demographics | **Absent** — no age, gender, income | Nothing to protect; also bounds the fairness audit to geography |

**Purpose limitation:** public Kaggle dataset released by Olist for analysis and education; this
use is within its stated purpose.

The PII exposure that actually matters here is the **free-text corpus**, because the deliverable
is published to a public GitHub repo. The mitigation is structural: text findings are reported as
aggregate frequencies and extracted n-grams, never as quoted reviews.

In [9]:
pii_flags = {
    "phone_like": reviews.review_comment_message.dropna().str.contains(r"\b\d{8,}\b").sum(),
    "email_like": reviews.review_comment_message.dropna().str.contains(r"\S+@\S+").sum(),
    "url_like": reviews.review_comment_message.dropna().str.contains(r"http|www\.").sum(),
}
governance = pd.DataFrame([
    {"item": "Direct identifiers present", "value": "No — surrogate hashes only"},
    {"item": "Geographic granularity", "value": "zip prefix + city + state (coarse)"},
    {"item": "Demographics present", "value": "No"},
    {"item": "Free-text fields", "value": f"{reviews.review_comment_message.notna().sum():,} comments"},
    *[{"item": f"Comments matching {k}", "value": f"{v:,}"} for k, v in pii_flags.items()],
    {"item": "Verbatim text in deliverable", "value": "Prohibited — aggregates and n-grams only"},
])
display(governance)
governance.to_csv(RAW / "_governance.csv", index=False)

,item,value
0,Direct identifiers present,No — surrogate hashes only
1,Geographic granularity,zip prefix + city + state (coarse)
2,Demographics present,No
3,Free-text fields,"40,977 comments"
4,Comments matching phone_like,0
5,Comments matching email_like,0
6,Comments matching url_like,2
7,Verbatim text in deliverable,Prohibited — aggregates and n-grams only


## 1.7 Write the immutable raw copy

Files are copied byte-for-byte from `archive/` to `data/raw/`. **Never edited after this point.**

In [10]:
for name in TABLES:
    shutil.copy2(ARCHIVE / f"{name}.csv", RAW / f"{name}.csv")

manifest = {
    "ingested_at_utc": pulled_at,
    "encoding": ENCODING,
    "encoding_rationale": (
        "latin-1 verified: utf-8 reads without error but silently corrupts accented "
        "Portuguese characters; cp1252 raises on byte 0x8f."
    ),
    "n_tables": len(TABLES),
    "total_rows": int(sum(len(v) for v in raw.values())),
    "random_state": RANDOM_STATE,
    "date_columns_parsed": {k: v for k, v in DATE_COLS.items()},
}
(RAW / "_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Copied {len(TABLES)} files to data/raw/ (immutable)")
print(f"Total rows ingested: {manifest['total_rows']:,}")

Copied 9 files to data/raw/ (immutable)
Total rows ingested: 1,550,922


## Stage 1 — Gate Checklist

Per `DOCS/STRUCTURE.md` §1. Every box below is backed by an artifact in this notebook.

- [x] **Raw data saved to `data/raw/`** (untouched copy) — §1.7, 9 files copied byte-for-byte
- [x] **Source, timestamp, and row/column counts logged** — `_lineage.csv` with SHA-256 per file
- [x] **Schema matches expectations** — 13/13 business rules passed (§1.5); 3 known deviations
      documented with their resolution, not silently accepted
- [x] **PII / data governance review completed** — §1.6, `_governance.csv`; free-text corpus
      identified as the material exposure, mitigation is structural (no verbatim quotes)
- [x] **Encoding verified and asserted** — latin-1, with the UTF-8 silent-corruption failure
      mode demonstrated in §1.1 rather than assumed

**Gate status: PASSED** → proceed to `02_cleaning.ipynb`.

---
### Changelog
*Pass 1* — initial ingestion. No loop-backs yet.

In [11]:
print("Stage 1 complete.")
print(f"  data/raw/          {len(list(RAW.glob('*.csv')))} csv + 4 log files")
print(f"  encoding           {ENCODING} (asserted)")
print("  next               02_cleaning.ipynb")

Stage 1 complete.
  data/raw/          12 csv + 4 log files
  encoding           latin-1 (asserted)
  next               02_cleaning.ipynb
